# Cable news chyrons

Associated chart: the chyron ternary and the Griner-swap scrolly

Output files:
`chyrons.json` in `src/data/`.

Input files: `data/chyron_matches.tsv`, `data/chyron_monthly.json` (both harvested from the Internet Archive's Third Eye project).


## Setup

In [ ]:
import json, math, re
from collections import Counter, defaultdict
from pathlib import Path

DATA = Path("../data")
OUT = Path("../src/data")

STOP = set("""a an the and or but of in on at to for with by from as is are was were be been
being it its this that these those i you he she they we his her their our your my me him them
us new says said say after amid over vs vs. into out up down off not no what who why how when
where which while will would could should can may might must have has had do does did done
about more most so than then just also only even still back get gets got make makes made
""".split())

def tokenize(text):
    text = re.sub(r"[‘’']s\b", "", text.lower())
    return re.findall(r"[a-z][a-z-]+", text)

CH_MAP = {"FOXNEWSW": "Fox News", "CNNW": "CNN", "MSNBCW": "MSNBC", "MSNOW": "MSNBC"}
channels = ["Fox News", "CNN", "MSNBC"]

chy = json.loads((DATA / "chyron_monthly.json").read_text())
MO_RE = re.compile(r"^20\d{2}-\d{2}$")
tot = defaultdict(Counter)
mat = defaultdict(Counter)
for raw_ch, label in CH_MAP.items():
    for mo, n in chy.get("totals", {}).get(raw_ch, {}).items():
        if MO_RE.match(mo): tot[label][mo] += n
    for mo, n in chy.get("matches", {}).get(raw_ch, {}).items():
        if MO_RE.match(mo): mat[label][mo] += n

all_months = sorted(set().union(*[set(t) for t in tot.values()]))
def nxt_mo(m):
    y, mm = int(m[:4]), int(m[5:7])
    return f"{y + (mm == 12)}-{(mm % 12) + 1:02d}"
axis, cur = [], all_months[0]
while cur <= all_months[-1]:
    axis.append(cur)
    cur = nxt_mo(cur)
print(f"{axis[0]} → {axis[-1]} · {len(axis)} months")

## Word counts per network

A Third Eye row often bundles several rotating banners; only the SEGMENTS that themselves
match a WNBA term are kept, so unrelated banners and OCR garble never enter the counts.
Player-word associations are per segment: a word only counts for a player when it
appears in the same banner as her name, not just the same screen. Surnames only — "ANGEL" alone could be Angel McCoughtry.


In [ ]:
CHY_DROP = set("""wnba caitlin clark angel reese brittney griner sophie cunningham
    paige bueckers aja a'ja wilson diana taurasi sue bird candace parker sabrina
    ionescu breanna stewart napheesa collier kelsey plum maya moore skylar diggins
    aliyah boston cameron brink dijonai carrington kamilla cardoso chennedy carter
    indiana fever chicago sky atlanta dream star stars player players""".split()) | STOP
CHY_NAMES = {"caitlin", "clark", "angel", "reese", "brittney", "griner",
             "sophie", "cunningham", "paige", "bueckers", "dijonai",
             "carrington", "diana", "taurasi", "sue", "bird", "candace",
             "parker", "sabrina", "ionescu", "breanna", "stewart",
             "napheesa", "collier", "kelsey", "plum", "maya", "moore",
             "skylar", "diggins", "aliyah", "boston", "cameron", "brink",
             "kamilla", "cardoso", "chennedy", "carter", "wilson"}
# a banner segment counts as WNBA when it contains any of these
WNBA_TERMS = [r"\bwnba\b", "caitlin clark", "angel reese", "brittney griner", r"\bgriner\b",
    "sophie cunningham", "paige bueckers", r"a'?ja wilson", "diana taurasi", "sue bird",
    "candace parker", "sabrina ionescu", "breanna stewart", "napheesa collier",
    "kelsey plum", "maya moore", "skylar diggins", "aliyah boston", "cameron brink",
    "dijonai carrington", "kamilla cardoso", "chennedy carter",
    "indiana fever", "chicago sky", "atlanta dream"]
SEG_RE = re.compile("|".join(WNBA_TERMS), re.I)
PLAYER_NAME_SETS = {
    "Caitlin Clark": {"clark"},
    "Angel Reese": {"reese"},
    "Brittney Griner": {"griner"},
    "Sophie Cunningham": {"cunningham"},
}
ERAS = ("pre", "post")  # before 2023 / 2023 onward

ex = defaultdict(dict)
words = defaultdict(Counter)
word_ex = defaultdict(dict)
adj_words = defaultdict(Counter)
adj_ex = defaultdict(dict)
pl_cooc = {p: defaultdict(Counter) for p in PLAYER_NAME_SETS}
pl_totals = {p: Counter() for p in PLAYER_NAME_SETS}
pl_cooc_era = {e: {p: defaultdict(Counter) for p in PLAYER_NAME_SETS} for e in ERAS}
pl_totals_era = {e: {p: Counter() for p in PLAYER_NAME_SETS} for e in ERAS}
words_era = {e: defaultdict(Counter) for e in ERAS}
pl_monthly = {p: defaultdict(Counter) for p in PLAYER_NAME_SETS}
mat_strict = defaultdict(Counter)
kept_of = lambda toks: [t for t in toks
                        if (t not in CHY_NAMES and t not in CHY_DROP and len(t) > 2
                            and not t.endswith("-") and t not in {"com", "www", "cnn", "fox", "msnbc"})]

for line in open(DATA / "chyron_matches.tsv"):
    parts = line.rstrip("\n").split("\t")
    if len(parts) < 5 or parts[0].startswith("date_"):
        continue
    label = CH_MAP.get(parts[1])
    if not label:
        continue
    mo = parts[0][:7]
    text = parts[4].replace("\\n", " / ")
    # advertisement chyrons (e.g. bedding ads citing a former WNBA player)
    if re.search(r"sheex|marciniak|promo code|\.com/", text, re.I):
        continue
    segs = [s.strip() for s in re.split(r"\s*/\s*", text) if s.strip()]
    wnba_segs = [s for s in segs if SEG_RE.search(s)]
    if not wnba_segs:
        continue
    mat_strict[label][mo] += 1
    text_w = " / ".join(wnba_segs)
    ex[label].setdefault(mo, text_w[:160])
    era = "pre" if mo < "2023" else "post"
    for t in set(tokenize(text_w)):
        if len(t) > 2 and not t.endswith("-") and t not in {"com", "www", "cnn", "fox", "msnbc"}:
            if t not in CHY_DROP:
                words[label][t] += 1
                words_era[era][label][t] += 1
                word_ex[t].setdefault(label, text_w[:150])
    hit_players = set()
    for s in wnba_segs:
        s_toks = set(tokenize(s))
        kept = kept_of(s_toks)
        if any(t in CHY_NAMES for t in s_toks):
            for t in kept:
                adj_words[label][t] += 1
                adj_ex[label].setdefault(t, s[:150])
        for player, names in PLAYER_NAME_SETS.items():
            if names & s_toks:
                hit_players.add(player)
                for t in kept:
                    pl_cooc[player][label][t] += 1
                    pl_cooc_era[era][player][label][t] += 1
    for player in hit_players:
        pl_totals[player][label] += 1
        pl_totals_era[era][player][label] += 1
        pl_monthly[player][label][mo] += 1

# the monthly matched series uses the stricter segment-level counts
if sum(sum(c.values()) for c in mat_strict.values()):
    mat = mat_strict
print({c: sum(mat[c].values()) for c in channels})

## The ternary and the output

Each word's share across the three networks, volume-adjusted: a word sits nearest the
network that says it most often as a share of that network's own WNBA chyrons, so the
bigger captioner doesn't win every word by default.


In [ ]:
ch_word_totals = {c: sum(words[c].values()) for c in channels}
all_w = Counter()
for c in channels:
    all_w.update(words[c])
tern_terms = []
for term, total in all_w.most_common(600):
    if total < 15:
        break
    raw_c = [words[c][term] for c in channels]
    rates = [n / max(ch_word_totals[c], 1) for n, c in zip(raw_c, channels)]
    ssum = sum(rates)
    if ssum == 0:
        continue
    shares = [round(r / ssum, 4) for r in rates]
    dom = channels[shares.index(max(shares))]
    ex_ch = dom if word_ex[term].get(dom) else next(iter(word_ex[term]), "")
    tern_terms.append({"term": term, "total": total, "counts": raw_c,
                       "shares": shares, "ex_ch": ex_ch,
                       "ex": word_ex[term].get(ex_ch, "")})

payload = {
    "months": axis,
    "channels": channels,
    "series": {c: {"matches": [mat[c].get(m, 0) for m in axis],
                   "totals": [tot[c].get(m, 0) for m in axis],
                   "ex": {m: ex[c].get(m, "") for m in axis if ex[c].get(m)}}
               for c in channels},
    "words": {c: [{"w": w, "c": n} for w, n in words[c].most_common(18)] for c in channels},
    "cooc": {c: (lambda ch: [
        {"w": w, "c": n,
         "wt": round(n * max(0.0, math.log(
             (n / max(sum(adj_words[ch].values()), 1)) /
             max(words[ch].get(w, n) / max(sum(words[ch].values()), 1), 1e-9))), 2),
         "ex": adj_ex[ch].get(w, "")}
        for w, n in adj_words[ch].most_common(200) if n >= 3
    ])(c) for c in channels},
    "ternary": tern_terms,
    "player_cooc": {p: {
        c: [{"w": w, "c": n,
             # how much more often the word appears in this player's chyrons
             # than in the channel's other WNBA chyrons
             "wt": round(n * max(0.0, math.log(
                 (n / max(sum(pl_cooc[p][c].values()), 1)) /
                 max((words[c].get(w, n) - n + 0.5) /
                     max(sum(words[c].values()) - sum(pl_cooc[p][c].values()), 1), 1e-9))), 2)}
            for w, n in pl_cooc[p][c].most_common(200) if n >= 3]
        for c in channels}
        for p in pl_cooc},
    "player_chyrons": {p: {c: pl_totals[p][c] for c in channels} for p in pl_totals},
    "player_cooc_era": {e: {p: {
        c: [{"w": w, "c": n,
             "wt": round(n * max(0.0, math.log(
                 (n / max(sum(pl_cooc_era[e][p][c].values()), 1)) /
                 max((words_era[e][c].get(w, n) - n + 0.5) /
                     max(sum(words_era[e][c].values()) - sum(pl_cooc_era[e][p][c].values()), 1), 1e-9))), 2)}
            for w, n in pl_cooc_era[e][p][c].most_common(200) if n >= 3]
        for c in channels}
        for p in pl_cooc_era[e]} for e in ERAS},
    "player_chyrons_era": {e: {p: {c: pl_totals_era[e][p][c] for c in channels}
                               for p in pl_totals_era[e]} for e in ERAS},
    "player_monthly": {p: {c: [pl_monthly[p][c].get(m, 0) for m in axis] for c in channels}
                       for p in pl_monthly},
}

#### Overwrites `src/data/chyrons.json`

In [ ]:
(OUT / "chyrons.json").write_text(json.dumps(payload))
griner = {t["term"]: t["shares"] for t in tern_terms if t["term"] in ("viktor", "bout", "merchant", "death")}
print(f"{len(tern_terms)} ternary terms · wrote chyrons.json")
griner

## Spot check

A fresh random sample each run — eyeball that the method is doing what it claims.


In [ ]:
import random

print("— random ternary words, with a real chyron each")
for t in random.sample(tern_terms, 5):
    lean = channels[t["shares"].index(max(t["shares"]))]
    print(f"   {t['term']!r} leans {lean} {t['shares']}")
    if t["ex"]: print(f"      “{t['ex'][:90]}”")

player = random.choice(list(pl_cooc))
ch = random.choice(channels)
words_here = pl_cooc[player][ch].most_common(30)
print(f"\n— random {ch} words sharing a banner with {player}")
for w, n in random.sample(words_here, min(4, len(words_here))):
    print(f"   {w} ({n})  “{adj_ex[ch].get(w, '')[:70]}”")